<a href="https://colab.research.google.com/github/HaoDing12/HaoDing12/blob/main/PageRank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PageRank

In this exercise, you will compute PageRank on a collection of 469,235 web sites using the iterative version of the PageRank algorithm described in class for sparse graphs.  As we discussed in class, this avoids performing cubic-time operations on a graph with a large number of nodes.

Furthermore, you should take advantage of the _sparsity_ of the original hyperlinks, even though "teleportation" or random jumps connect every note to every other node.  As long as the maximum degree of the original link graph is much less than the number of nodes, you should be able to keep each iteration's runtime less than quadratic in the number of nodes.

Consider the following directed graph:

![A directed link graph](https://www.khoury.northeastern.edu/home/dasmith/courses/cs6200//pagerank.jpg)

We can represent this graph as a collection of nodes, here, ordered pairs of node index and node name:

In [6]:
small_nodes = [(0, 'A'),
              (1, 'B'),
              (2, 'C'),
              (3, 'D'),
              (4, 'E'),
              (5, 'F')]

and a collection of directed links, i.e., ordered pairs from source to target:

In [7]:
small_edges = [
  (0, 1),
  (0, 2),
  (0, 5),
  (1, 2),
  (1, 3),
  (1, 4),
  (1, 5),
  (2, 3),
  (2, 4),
  (3, 0),
  (3, 2),
  (3, 4),
  (3, 5),
  (4, 0),
  (5, 0),
  (5, 1),
  (5, 4)
]

We use integer identifiers for the nodes for efficiency. In most real-world collections of hyperlinks, unlike this example, not every page will have in-links, nor will every page have out-links.

## First Implementation and Test

\[10 points\] Implement the iterative PageRank algorithm we discussed in class. Test your code on the six-node example using the input representation given above.  Be sure that your code handles pages that have no in-links or out-links properly.  (You may wish to test on a few such examples.) In later parts of this assignment, depending on how you store the data, it may be convenient to use iterators rather than storing the data in memory.

In [8]:
# TODO: Implement PageRank, given nodes and edges, to start with a uniform
# distribution over nodes, run a fixed number of iterations, and
# return a distribution over nodes.

LAMBDA = 0.15   # random jump (teleportation) probability

def out_degrees(n, edges):
    deg = [0] * n
    for s, t in edges:
        deg[s] += 1
    return deg

def pr_step(I, edges, outdeg, n, lam=LAMBDA):
    """One PageRank iteration. Runs in O(nodes + edges), never O(nodes^2)."""
    # Every node gets the random-jump share.
    R = [lam / n] * n
    # Follow real links: each page splits (1 - lam) of its rank among its out-links.
    for s, t in edges:
        R[t] += (1 - lam) * I[s] / outdeg[s]
    # Sinks (no out-links) spread their rank evenly over all pages. Instead of adding
    # n edges per sink, add their total mass to every node once.
    sink_mass = sum(I[p] for p in range(n) if outdeg[p] == 0)
    add = (1 - lam) * sink_mass / n
    return [r + add for r in R]

def page_rank_fixed_iter(nodes, edges, iterations=10):
    n = len(nodes)
    outdeg = out_degrees(n, edges)
    I = [1 / n] * n   # start from the uniform distribution
    for _ in range(iterations):
        I = pr_step(I, edges, outdeg, n)
    return [(i, name, I[i]) for i, name in nodes]



# Output PageRank on the toy graph at various points.
# Make sure your output has node number, name, and PageRank value.
page_rank_fixed_iter(small_nodes, small_edges, 1)
page_rank_fixed_iter(small_nodes, small_edges, 10)
page_rank_fixed_iter(small_nodes, small_edges, 100)

[(0, 'A', 0.25212710537519556),
 (1, 'B', 0.13930618531853856),
 (2, 'C', 0.15130648986670525),
 (3, 'D', 0.11890782257353917),
 (4, 'E', 0.18704590699931606),
 (5, 'F', 0.15130648986670525)]

In [10]:
# Edge cases: pages with no out-links (sinks), no in-links, isolated pages, self-links.
tests = {
    "A->B, C isolated (B is a sink; A and C have no in-links)":
        ([(0, 'A'), (1, 'B'), (2, 'C')], [(0, 1)]),
    "no links at all (every page is a sink)":
        ([(0, 'A'), (1, 'B'), (2, 'C')], []),
    "self-link A->A plus A->B (B is a sink)":
        ([(0, 'A'), (1, 'B')], [(0, 0), (0, 1)]),
}
for desc, (nodes, edges) in tests.items():
    result = page_rank_fixed_iter(nodes, edges, 100)
    total = sum(pr for _, _, pr in result)
    print(desc)
    print("  ", [(name, round(pr, 4)) for _, name, pr in result], "sum =", round(total, 6))

A->B, C isolated (B is a sink; A and C have no in-links)
   [('A', 0.2597), ('B', 0.4805), ('C', 0.2597)] sum = 1.0
no links at all (every page is a sink)
   [('A', 0.3333), ('B', 0.3333), ('C', 0.3333)] sum = 1.0
self-link A->A plus A->B (B is a sink)
   [('A', 0.5), ('B', 0.5)] sum = 1.0


## PageRank on Web Crawl Data

\[20 points\] Download and unpack a list of `.edu` websites and the links among them from the [Common Crawl](https://commoncrawl.org/2017/05/hostgraph-2017-feb-mar-apr-crawls/) open-source web crawl. For the sake of brevity, the data record links among websites, not web pages. The information for nodes and links is the same as the toy example above.

In [9]:
# If you're running on a machine (e.g., Windows) that doesn't have wget or gzip,
# feel free to comment this out and use a different set of commands to load
# the data.
!wget https://khoury.northeastern.edu/home/dasmith/courses/cs6200/vertices-edu.txt.gz
!gzip -df vertices-edu.txt.gz
!wget https://khoury.northeastern.edu/home/dasmith/courses/cs6200/edges-edu.txt.gz
!gzip -df edges-edu.txt.gz

--2026-10-09 01:53:57--  https://khoury.northeastern.edu/home/dasmith/courses/cs6200/vertices-edu.txt.gz
Resolving khoury.northeastern.edu (khoury.northeastern.edu)... 52.70.229.197
Connecting to khoury.northeastern.edu (khoury.northeastern.edu)|52.70.229.197|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3703486 (3.5M) [application/x-gzip]
Saving to: ‘vertices-edu.txt.gz’

vertices-edu.txt.gz 100%[===================>]   3.53M  2.54MB/s    in 1.4s    

2026-10-09 01:54:00 (2.54 MB/s) - ‘vertices-edu.txt.gz’ saved [3703486/3703486]

--2026-10-09 01:54:00--  https://khoury.northeastern.edu/home/dasmith/courses/cs6200/edges-edu.txt.gz
Resolving khoury.northeastern.edu (khoury.northeastern.edu)... 52.70.229.197
Connecting to khoury.northeastern.edu (khoury.northeastern.edu)|52.70.229.197|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12829526 (12M) [application/x-gzip]
Saving to: ‘edges-edu.txt.gz’

edges-edu.txt.gz    100%[===========

There should now be files `vertices-edu.txt` and `edges-edu.txt`.

In [11]:
# TODO: Process the raw data into the same format as the simple graph.
# You may create lists or iterators.
# Nodes: one "id name" pair per line, same format as small_nodes.
nodes = []
with open('vertices-edu.txt') as f:
    for line in f:
        i, name = line.split()
        nodes.append((int(i), name))

# Edges: one "source target" pair per line, same format as small_edges.
edges = []
with open('edges-edu.txt') as f:
    for line in f:
        s, t = line.split()
        edges.append((int(s), int(t)))

# pr_step indexes rank lists by node id, so ids must be exactly 0..n-1.
assert all(i == k for k, (i, _) in enumerate(nodes))

print(len(nodes), len(edges))
print(nodes[:3], edges[:3])

469235 3300462
[(0, 'edu.00zl5e'), (1, 'edu.06hxbt'), (2, 'edu.082ifc')] [(386, 440), (19202, 1033), (103884, 2635)]


Refine your implementation of PageRank to test for numerical convergence. Specificially, at each iteration, calculate the [perplexity](https://en.wikipedia.org/wiki/Perplexity) of the PageRank distribution, where perplexity is defined as 2 raised to the [Shannon entropy](https://en.wikipedia.org/wiki/Entropy_(information_theory)) of the PageRank distribution, i.e., $2^{H(PR)}$. (Recall the definition of entropy from our discussion of data compression.) The maximum perplexity of a PageRank distribution will therefore be the number of nodes in the graph.

At each iteration, check the _change_ in perplexity. If the change is less than some threshold, you can stop.


In [12]:
# TODO: Implement convergence testing in PageRank
# If you choose, you can share some subroutines with your first version.
# Print the change in perplexity at each iteration.

import math

def perplexity(I):
    # 2 ** (Shannon entropy in bits). Equals n for the uniform distribution.
    H = -sum(p * math.log2(p) for p in I if p > 0)
    return 2 ** H

def page_rank(nodes, edges, threshold=1):
    n = len(nodes)
    outdeg = out_degrees(n, edges)
    I = [1 / n] * n   # start from the uniform distribution
    old = perplexity(I)
    it = 0
    while True:
        I = pr_step(I, edges, outdeg, n)   # same iteration as page_rank_fixed_iter
        it += 1
        new = perplexity(I)
        change = old - new
        print(f"iter {it}: perplexity {new:.2f}, change {change:.2f}")
        if abs(change) < threshold:
            break
        old = new
    return [(i, name, I[i]) for i, name in nodes]

# Run until perplexity changes by less than 1
PR = page_rank(nodes, edges, 1)



iter 1: perplexity 298071.45, change 171163.55
iter 2: perplexity 255742.81, change 42328.63
iter 3: perplexity 234252.27, change 21490.54
iter 4: perplexity 225975.96, change 8276.32
iter 5: perplexity 221539.00, change 4436.96
iter 6: perplexity 219533.96, change 2005.04
iter 7: perplexity 218300.92, change 1233.04
iter 8: perplexity 217671.94, change 628.98
iter 9: perplexity 217238.01, change 433.92
iter 10: perplexity 216999.38, change 238.63
iter 11: perplexity 216815.23, change 184.15
iter 12: perplexity 216708.53, change 106.70
iter 13: perplexity 216619.07, change 89.46
iter 14: perplexity 216564.52, change 54.55
iter 15: perplexity 216516.23, change 48.29
iter 16: perplexity 216485.35, change 30.88
iter 17: perplexity 216457.06, change 28.29
iter 18: perplexity 216438.19, change 18.87
iter 19: perplexity 216420.60, change 17.59
iter 20: perplexity 216408.43, change 12.17
iter 21: perplexity 216397.01, change 11.42
iter 22: perplexity 216388.86, change 8.14
iter 23: perplexity

## Link Analysis

\[20 points\] In this final section, you will compute some properties of this web-site graph.

First, consider the _in-link count_ of a website, simply the number of websites pointing to it (including self-links).

In [13]:
# TODO: List the document ID, domain name, and in-link count of the 70 websites with the highest in-link count
n = len(nodes)
indeg = [0] * n
for s, t in edges:
    indeg[t] += 1   # the edge file has no duplicate edges, so this counts distinct linking sites

top_in = sorted(range(n), key=lambda i: indeg[i], reverse=True)[:70]
for i in top_in:
    print(i, nodes[i][1], indeg[i])

185524 edu.mit.web 4388
278032 edu.stanford 4021
244433 edu.purdue.english.owl 3531
140443 edu.indiana 3339
237176 edu.princeton 3251
64587 edu.columbia 3123
465503 edu.yale 2804
418623 edu.utexas 2622
383763 edu.unc 2592
197698 edu.nap 2494
439637 edu.washington 2291
373442 edu.umich 2281
440674 edu.washington.depts 2276
148945 edu.jhu.muse 2255
60975 edu.colorado 2232
449738 edu.wisc 2230
38320 edu.bu 2205
83572 edu.dartmouth 1965
408380 edu.usc 1952
178879 edu.mit 1946
27307 edu.berkeley 1908
233405 edu.pitt 1857
191069 edu.msu 1810
326371 edu.uchicago.press 1763
136464 edu.illinois 1753
93874 edu.educause 1741
56979 edu.cmu.cs 1730
199032 edu.ncsu 1709
36294 edu.brown 1702
202182 edu.nd 1689
68675 edu.cornell 1685
71095 edu.cornell.law 1646
183214 edu.mit.mitpress 1644
215627 edu.nyu 1625
56538 edu.cmu 1583
239378 edu.psu 1541
350412 edu.ufl 1533
120819 edu.harvard 1529
270369 edu.si 1513
107916 edu.gatech 1500
365396 edu.uky 1497
337138 edu.ucop 1482
358246 edu.uic 1472
382564 edu

Then, use the PageRank values compute by your second implementation. Note that some websites will have both a high in-link count and PageRank.

In [14]:
# TODO: List the document ID, domain name, and PageRank of the 70 websites with the highest PageRank.
top_pr = sorted(PR, key=lambda x: x[2], reverse=True)[:70]
for i, name, pr in top_pr:
    print(i, name, f"{pr:.6g}")

185524 edu.mit.web 0.000928005
465503 edu.yale 0.00070059
318278 edu.ucdavis.cas 0.000641567
278032 edu.stanford 0.000594467
136464 edu.illinois 0.000579699
319795 edu.ucdavis.ice.code 0.000546531
237176 edu.princeton 0.000501956
284517 edu.stanford.web 0.000474784
383763 edu.unc 0.000471677
64587 edu.columbia 0.000447368
449738 edu.wisc 0.000436207
60975 edu.colorado 0.000432396
346969 edu.ucsf 0.00043005
140443 edu.indiana 0.000429624
42502 edu.byu.cas 0.00041783
334739 edu.uconn 0.000412787
14945 edu.arizona 0.000409276
244433 edu.purdue.english.owl 0.000403029
373442 edu.umich 0.000402079
439637 edu.washington 0.000398425
408380 edu.usc 0.000391455
191069 edu.msu 0.000391395
281817 edu.stanford.med 0.000391387
418623 edu.utexas 0.000381956
239378 edu.psu 0.000378577
68675 edu.cornell 0.000373596
342997 edu.ucsd 0.000357974
153823 edu.kit 0.000357911
293521 edu.tamu 0.000356617
197698 edu.nap 0.000353078
337138 edu.ucop 0.000345795
233405 edu.pitt 0.000344177
307291 edu.ttu.depts 0.

Finally, compute some summary statistics on this dataset.

In [16]:
# TODO: Compute:
outdeg = out_degrees(n, edges)
# - the proportion of websites with no in-links (i.e., source nodes);
print("no in-links (sources):", sum(1 for d in indeg if d == 0) / n)
# - the proportion of websites with no out-links (i.e., sink nodes);
print("no out-links (sinks):", sum(1 for d in outdeg if d == 0) / n)
# - the proportion of websites whose PageRank is higher than the initial uniform distribution.
print("PageRank above uniform 1/n:", sum(1 for _, _, pr in PR if pr > 1 / n) / n)


no in-links (sources): 0.26153633041013563
no out-links (sinks): 0.6106641661427643
PageRank above uniform 1/n: 0.15514614212494807
